# 04 — Pilot: LLM/RAG trả lời theo phiên bản nào?

**Đề tài:** Trôi phiên bản tri thức trong trợ lý CTI dùng LLM/RAG

| Notebook | Việc | Đầu ra |
|---|---|---|
| `01_thay_doi_phien_ban_ATTACK` | Ground truth các thay đổi giữa các phiên bản ATT&CK | `drift/` |
| `02_benchmark_theo_phien_ban` | Benchmark hỏi–đáp cho một phiên bản (chạy cho v19.2 và v18.1) | `benchmark/attack-<phiên bản>/` |
| `03_cap_cau_hoi_truoc_sau` | Cặp câu hỏi trước/sau có nhãn kiểu thay đổi, kho RAG trộn | `benchmark/drift-18.1-19.2/` |
| **`04_pilot_llm_rag` (notebook này)** | Cho LLM trả lời 1.250 truy vấn pilot trong 3 điều kiện, chấm điểm, quyết định đi tiếp hay dừng | `results/pilot-18.1-19.2/` |

## Pilot để làm gì?

Trước khi đầu tư vài tháng cho thí nghiệm đầy đủ, ta cần biết **hiện tượng có thật và đủ lớn không**:

1. Khi tri thức ATT&CK đã đổi, LLM có **trả lời theo bản cũ** (STALE) nhiều hơn hẳn so với các câu không đổi không?
2. **RAG có chữa được không?** RAG chỉ dùng bản mới có thể vẫn thất bại với mã đã bị thu hồi (notebook 03 cho thấy kho bản mới không còn nhắc mã cũ). RAG trộn hai phiên bản có thể **gây nhiễu** thêm.
3. Tỉ lệ lỗi có **khác nhau theo kiểu thay đổi** không (đổi tactic, mã bị thu hồi, cùng mã đổi tên…)?

Nếu cả ba câu trả lời đều rõ ràng, đề tài **đi tiếp** (GO). Nếu không, ta điều chỉnh trước khi tốn thêm công sức.

## Thiết kế

| Thành phần | Lựa chọn | Lý do |
|---|---|---|
| Truy vấn | 1.250 truy vấn pilot của notebook 03 (435 mục × tối đa 3 cách hỏi) | đủ mọi lớp thay đổi, có nhóm đối chứng `STABLE` |
| Mô hình | 2 mô hình mở, khác họ, **phát hành trước ATT&CK v19 (28/4/2026)**, lượng tử hoá 4-bit, không cần đăng ký | đo đúng tình huống "mô hình chỉ biết bản cũ"; chạy vừa RTX 4070 12 GB |
| Điều kiện | `no_rag` (chỉ tri thức của mô hình) · `rag_new` (kho v19.2) · `rag_mixed` (kho trộn v18.1 + v19.2) | ba tình huống triển khai thực tế |
| Truy hồi | BM25, top-5 đoạn | tất định, không tốn GPU, bám mã định danh (T1562…) tốt |
| Giải mã | tham lam (`temperature = 0`), **ép đầu ra JSON** `{"answer": [...]}` bằng *structured outputs* của vLLM | chấm điểm không phải đoán; tái lập được |
| Chấm điểm | `ctidrift_scoring.py` (notebook 03) | một định nghĩa chỉ số duy nhất |

**Mô hình:**

| Nhãn | Kho Hugging Face | Phát hành | Ghi chú |
|---|---|---|---|
| `qwen3-8b` | `Qwen/Qwen3-8B-AWQ` | 05/2025 | tắt chế độ "suy nghĩ" (`enable_thinking=False`) |
| `llama3.1-8b` | `hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4` | 07/2024 | bản lượng tử hoá AWQ của Llama 3.1 8B Instruct (giấy phép Llama 3.1 Community) |

Mỗi mô hình được **ghim đúng phiên bản** (mã commit trên Hugging Face). Ai chạy lại cũng dùng đúng trọng số đó.

**Khác biệt đã biết giữa hai mô hình:** mẫu hội thoại (chat template) mặc định của Llama 3.1 tự chèn dòng *"Cutting Knowledge Date: December 2023 / Today Date: 26 Jul 2024"*. Qwen3 không chèn ngày. Ta giữ nguyên mẫu mặc định của mỗi mô hình, đúng cách người dùng thật triển khai, và ghi nhận khác biệt này khi so sánh hai mô hình.

## Lưu ý phương pháp (đọc trước khi diễn giải kết quả)

- **Mục pilot là mục "đã dùng".** 435 mục pilot gồm cả mục thuộc tập dev lẫn test. Mọi quyết định thiết kế rút ra từ pilot (prompt, top-k, phương pháp Trụ 3) đều đã "nhìn" các mục này. Vì vậy danh sách mục pilot được ghi vào manifest. **Kết quả cuối cùng trong bài báo sẽ báo cáo trên các mục test KHÔNG nằm trong pilot.**
- **Một sự kiện chi phối nhóm D1.** Phần lớn câu D1 có đáp án đổi đến từ một sự kiện duy nhất: tách *Defense Evasion* thành TA0005 *Stealth* và TA0112 *Defense Impairment*. Mô hình phát hành trước v19 **không thể biết** TA0112 nếu không có RAG. Vì vậy tỉ lệ lỗi thời cao ở điều kiện `no_rag` là **điều được dự đoán**, không phải phát hiện. Phát hiện thật nằm ở câu hỏi: **khi RAG đã đưa đúng bằng chứng mới vào ngữ cảnh, mô hình có vẫn trả lời theo bản cũ không** (xung đột tri thức), và **kho trộn có làm tình hình tệ hơn không**.
- **Khoảng tin cậy theo cụm.** Một mục sinh ra 2–3 truy vấn (các cách hỏi), nên các truy vấn không độc lập. Ngoài khoảng Wilson (coi từng truy vấn là độc lập), notebook tính **khoảng tin cậy bootstrap theo cụm mục** (`item_id`). Tiêu chí đi tiếp / dừng chỉ dùng khoảng bootstrap.
- **Câu D4 chứa sẵn mã cũ** ("…is T1562.004 still valid? If yes, repeat the ID…"). Ở `no_rag`, mô hình không biết mã đã bị thu hồi nên gần như chắc chắn lặp lại mã cũ. Đây đúng là thất bại ngoài thực tế (trợ lý "xác nhận" một thẻ đã lỗi thời), nhưng **không dùng D4 làm bằng chứng cho xung đột tri thức**.
- **So sánh `unversioned` và `current`** đo tác động của **cả** việc nói rõ phiên bản **lẫn** câu gợi ý "nếu mã đã bị thay thế thì trả lời theo mã mới" (chỉ có ở `current`). Hai yếu tố này sẽ được tách riêng ở thí nghiệm chính.

## Chạy notebook này thế nào?

1. Mở terminal, `conda activate cti-drift`, rồi `jupyter lab` trong `~/p/cti-drift/`.
2. Chạy **lần lượt từng ô**. Ô suy luận (mục 7) chạy lâu, khoảng **30–60 phút cho mỗi mô hình**. Trong lúc chờ, em theo dõi tiến độ bằng lệnh `tail -f` mà notebook in ra.
3. Nếu bị ngắt giữa chừng (mất điện, đóng máy), em **chạy lại notebook từ đầu**: phần đã xong được giữ lại, chỉ phần còn thiếu chạy tiếp.
4. Ô cuối tạo file `gui_thay.zip`. Em gửi thầy file đó cùng notebook đã chạy.

### Nhật ký phiên bản

| Phiên bản | Ngày | Thay đổi |
|---|---|---|
| 1.0 | 12/10/2026 | Bản đầu tiên |
| 1.1 | 12/10/2026 | Sửa theo kiểm tra độc lập: JSON dạng gọn (không khoảng trắng thừa); `MAX_TOKENS` 160; chạy thử đủ 4 mẫu câu hỏi; khoá chạy gồm cả mã nguồn module và phiên bản vLLM; dừng ô thì tắt hẳn tiến trình GPU; khoảng tin cậy bootstrap theo cụm mục; tiêu chí đi tiếp / dừng thiết kế lại để không đạt được chỉ nhờ giới hạn tri thức; thêm bảng xung đột tri thức, kho trộn so với kho mới (ghép cặp), D1 theo kiểu tactic; ghi danh sách mục pilot. |

## 1. Cấu hình và kiểm tra đầu vào

Mọi đường dẫn tương đối với thư mục dự án. Mô hình và bộ nhớ đệm của vLLM nằm trên ổ `/data` (đã thiết lập), notebook tự đặt các biến môi trường cần thiết, nên **không phụ thuộc** việc Jupyter có đọc `~/.bashrc` hay không.

In [1]:
from pathlib import Path
import os, sys, json, re, math, time, hashlib, random, subprocess, collections, datetime, zipfile
import pandas as pd

V_OLD, V_NEW = "18.1", "19.2"
ROOT    = Path.cwd()
DRIFT   = ROOT / "benchmark" / f"drift-{V_OLD}-{V_NEW}"
OUT     = ROOT / "results" / f"pilot-{V_OLD}-{V_NEW}"
PROMPTS = OUT / "prompts"
RAW     = OUT / "raw"
LOGS    = OUT / "logs"
SMOKE_PROMPTS = OUT / "prompts_smoke"
for p in (OUT, PROMPTS, RAW, LOGS, SMOKE_PROMPTS):
    p.mkdir(parents=True, exist_ok=True)

# Ổ dữ liệu: mô hình Hugging Face và bộ nhớ đệm biên dịch của vLLM
HF_HOME         = "/data/hf"
VLLM_CACHE_ROOT = "/data/vllm-cache"

MODELS = [
    dict(tag="qwen3-8b", repo="Qwen/Qwen3-8B-AWQ",
         revision="4da05a8edb55c6046cce958586c33b61da07bb79", released="2025-05",
         chat_template_kwargs={"enable_thinking": False}),
    dict(tag="llama3.1-8b", repo="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
         revision="db1f81ad4b8c7e39777509fac66c652eb0a52f91", released="2024-07",
         chat_template_kwargs={}),
]
CONDITIONS      = ["no_rag", "rag_new", "rag_mixed"]
TOP_K           = 5         # số đoạn RAG đưa vào ngữ cảnh
MAX_CHUNK_WORDS = 250       # cắt mỗi đoạn để prompt không quá dài
MAX_MODEL_LEN   = 6144      # độ dài tối đa (token) của prompt + câu trả lời
MAX_TOKENS      = 160       # độ dài tối đa câu trả lời JSON (đủ cho 8 phần tử kèm tên)
GPU_MEM         = 0.85      # tỉ lệ bộ nhớ GPU vLLM được dùng
SEED            = 2026
SMOKE_PER_TPL   = 2         # chạy thử nhanh: số truy vấn mỗi mẫu câu hỏi (đủ 4 kiểu lược đồ JSON)
BOOT_B          = 2000      # số lần lặp bootstrap theo cụm
PILOT_VERSION   = "1.1"

REQUIRED = {
    "pilot_queries":  DRIFT / "pilot_queries.jsonl",
    "drift_manifest": DRIFT / "manifest.json",
    "tactic_alias":   DRIFT / "tactic_alias.json",
    "corpus_mixed":   DRIFT / "corpus_mixed.jsonl",
    "corpus_new":     ROOT / "benchmark" / f"attack-{V_NEW}" / "corpus.jsonl",
    "scoring_module": ROOT / "ctidrift_scoring.py",
}
missing = [f"{k}: {p}" for k, p in REQUIRED.items() if not p.exists()]
assert not missing, "Thiếu đầu vào — hãy chạy notebook 02 và 03 trước:\n  " + "\n  ".join(missing)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# Bộ pilot phải đúng là bộ notebook 03 đã ghi (so mã băm trong manifest của notebook 03)
DM = json.loads(REQUIRED["drift_manifest"].read_text())
assert DM["outputs"]["pilot_queries.jsonl"] == sha256(REQUIRED["pilot_queries"]), \
    "pilot_queries.jsonl không khớp manifest của notebook 03 — hãy chạy lại notebook 03"
print(f"Bộ pilot: notebook 03 phiên bản {DM['driftset_version']}, {DM['counts']['pilot_queries']} truy vấn")

ENV = {**os.environ, "HF_HOME": HF_HOME, "HF_HUB_CACHE": f"{HF_HOME}/hub", "VLLM_CACHE_ROOT": VLLM_CACHE_ROOT,
       "VLLM_USE_FLASHINFER_SAMPLER": "0",        # trình dịch CUDA của hệ thống quá cũ (11.8) — xem nhật ký cài đặt
       "TOKENIZERS_PARALLELISM": "false"}
ENV.pop("HF_HUB_OFFLINE", None)                # lần đầu cần tải mô hình
if not Path(HF_HOME).is_dir():
    print(f"⚠ Không thấy {HF_HOME} — kiểm tra ổ /data đã gắn chưa (lệnh: findmnt /data)")

MANIFEST = {"pilot_version": PILOT_VERSION, "v_old": V_OLD, "v_new": V_NEW,
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "config": dict(CONDITIONS=CONDITIONS, TOP_K=TOP_K, MAX_CHUNK_WORDS=MAX_CHUNK_WORDS,
                           MAX_MODEL_LEN=MAX_MODEL_LEN, MAX_TOKENS=MAX_TOKENS, GPU_MEM=GPU_MEM, SEED=SEED,
                           BOOT_B=BOOT_B),
            "models": MODELS,
            "inputs": {k: {"path": str(p.relative_to(ROOT)), "sha256": sha256(p)} for k, p in REQUIRED.items()}}
for k, v in MANIFEST["inputs"].items():
    print(f"{k:15s} {v['sha256'][:12]}…  {v['path']}")

Bộ pilot: notebook 03 phiên bản 1.2, 1250 truy vấn
pilot_queries   1382ad00e2b4…  benchmark/drift-18.1-19.2/pilot_queries.jsonl
drift_manifest  6abfab5da091…  benchmark/drift-18.1-19.2/manifest.json
tactic_alias    a5efc7448153…  benchmark/drift-18.1-19.2/tactic_alias.json
corpus_mixed    301ddae87bf7…  benchmark/drift-18.1-19.2/corpus_mixed.jsonl
corpus_new      0198ec96b46b…  benchmark/attack-19.2/corpus.jsonl
scoring_module  56b3cb0eb0c8…  ctidrift_scoring.py


## 2. Nạp truy vấn pilot và module chấm điểm

Mỗi truy vấn có `gold` (đáp án đúng theo phiên bản được hỏi), `other_gold` (đáp án theo phiên bản kia), `pair_class`, `framing`… (xem notebook 03).

In [2]:
import importlib, ctidrift_scoring
importlib.reload(ctidrift_scoring)
from ctidrift_scoring import score_drift, json_answer

QUERIES = [json.loads(l) for l in open(REQUIRED["pilot_queries"], encoding="utf-8")]
QBYID = {q["query_id"]: q for q in QUERIES}
assert len(QBYID) == len(QUERIES)
TACTIC_ALIAS = json.loads(REQUIRED["tactic_alias"].read_text())["alias"]
print(f"{len(QUERIES)} truy vấn | " + ", ".join(f"{k}={v}" for k, v in sorted(collections.Counter(q['template'] for q in QUERIES).items())))
pd.crosstab(pd.Series([q["pair_class"] for q in QUERIES], name="pair_class"),
            pd.Series([q["framing"] for q in QUERIES], name="framing"), margins=True)

1250 truy vấn | D1_tech_tactic=676, D2_subtech_parent=177, D3_tech_mitigation=214, D4_id_currency=183


framing,current,historical,unversioned,All
pair_class,,,,
ANCHOR_RENAMED,12,12,12,36
ANSWER_CHANGED,72,72,72,216
LABEL_CHANGED,80,80,80,240
NEW_ONLY,55,0,55,110
REVOKED_ANCHOR,56,56,56,168
STABLE,160,160,160,480
All,435,380,435,1250


## 3. Bộ truy hồi BM25

**BM25** là thuật toán tìm kiếm theo từ khoá chuẩn (nền tảng của Elasticsearch, Lucene). Ta tự cài đặt khoảng 30 dòng để kết quả **tất định** và không phụ thuộc thư viện ngoài. Bước tách từ giữ nguyên mã định danh: `T1562.004` thành một từ `t1562.004`, `TA0005` thành `ta0005`.

Hai kho được lập chỉ mục:

- `rag_new`: kho v19.2 của notebook 02 (đúng như một hệ thống đã cập nhật lên bản mới).
- `rag_mixed`: kho trộn của notebook 03 (hệ thống nạp thêm bản mới mà không xoá bản cũ). Những đoạn giống hệt nhau ở hai bản chỉ giữ một bản.

In [3]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:\.[0-9]+)*")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

import numpy as np

class BM25:
    def __init__(self, docs, k1=1.2, b=0.75):
        # docs: dict mã_đoạn -> văn bản. Đoạn được đánh số theo thứ tự mã (đã sắp xếp) -> hoà điểm xếp theo mã.
        self.ids = sorted(docs)
        n = len(self.ids)
        toks = [collections.Counter(tokenize(docs[i])) for i in self.ids]
        dl = np.array([sum(t.values()) for t in toks], dtype=float)
        avgdl = dl.mean()
        post = collections.defaultdict(lambda: ([], []))
        for j, tf in enumerate(toks):
            for t, c in tf.items():
                post[t][0].append(j)
                post[t][1].append(c)
        self.post = {}
        for t, (idx, tf) in post.items():
            idx, tf = np.array(idx), np.array(tf, dtype=float)
            idf = math.log(1 + (n - len(idx) + 0.5) / (len(idx) + 0.5))
            w = idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[idx] / avgdl))   # trọng số BM25 tính sẵn
            self.post[t] = (idx, w)
        self.n = n

    def search(self, query, k=5):
        scores = np.zeros(self.n)
        for t in sorted(set(tokenize(query))):          # thứ tự cộng cố định -> kết quả tất định
            if t in self.post:
                idx, w = self.post[t]
                scores[idx] += w
        cand = np.nonzero(scores)[0]
        order = cand[np.lexsort((cand, -scores[cand]))][:k]   # điểm giảm dần; hoà điểm: mã đoạn tăng dần
        return [(self.ids[j], float(scores[j])) for j in order]

def load_corpus(p):
    return {c["_id"]: c for c in map(json.loads, open(p, encoding="utf-8"))}

CORPORA = {"rag_new": load_corpus(REQUIRED["corpus_new"]), "rag_mixed": load_corpus(REQUIRED["corpus_mixed"])}
t0 = time.time()
INDEX = {c: BM25({i: d["title"] + "\n" + d["text"] for i, d in C.items()}) for c, C in CORPORA.items()}
print({c: len(C) for c, C in CORPORA.items()}, f"| lập chỉ mục {time.time() - t0:.1f}s")

# Kiểm thử nhanh: tìm theo mã và tên thì đoạn đứng đầu phải thuộc đúng tài liệu của mã đó
top = INDEX["rag_new"].search("ATT&CK technique T1055.011 Extra Window Memory Injection", 3)
assert CORPORA["rag_new"][top[0][0]]["metadata"]["doc_id"] == "T1055.011", top
assert INDEX["rag_new"].search("T1055.011", 1) == INDEX["rag_new"].search("T1055.011", 1)   # tất định
top_m = [i for i, _ in INDEX["rag_mixed"].search("T1222 File and Directory Permissions Modification tactic", 5)]
print("Ví dụ truy hồi kho trộn cho T1222:", top_m)

{'rag_new': 4738, 'rag_mixed': 5206} | lập chỉ mục 0.5s
Ví dụ truy hồi kho trộn cho T1222: ['18.1|T1222#0', '19.2|T1222#0', '18.1|T1222.002#0', '18.1|T1222.001#0', '19.2|T1222.002#0']


## 4. Truy hồi cho mọi truy vấn và chất lượng truy hồi

Câu hỏi được dùng nguyên văn làm truy vấn tìm kiếm. Với mỗi truy vấn ta ghi lại:

- `recall`: tỉ lệ **đoạn bằng chứng** (notebook 03) nằm trong top-5. Với `rag_new`, bằng chứng là đoạn của kho v19.2. Với `rag_mixed`, bằng chứng là đoạn đúng **phiên bản được hỏi**.
- `n_old` / `n_new`: số đoạn **chỉ có ở bản cũ** / **chỉ có ở bản mới** trong top-5 (kho trộn).
- `conflict`: top-5 chứa **cả hai phiên bản** của cùng một đoạn. Đây là tình huống mô hình nhận hai thông tin mâu thuẫn.

Bảng này là kết quả riêng của **tầng truy hồi**, chưa có LLM. Nó giúp giải thích các lỗi ở mục 9.

In [4]:
def evidence_for(q, cond):
    if cond == "rag_new":
        return q["evidence"][V_NEW] if q["target_version"] == V_NEW else []
    return q["evidence_mixed"][q["target_version"]]

RETR = {}
rows = []
for cond in ("rag_new", "rag_mixed"):
    C = CORPORA[cond]
    for q in QUERIES:
        hits = [i for i, _ in INDEX[cond].search(q["question"], TOP_K)]
        RETR[(cond, q["query_id"])] = hits
        ev = evidence_for(q, cond)
        vers = [tuple(C[h]["metadata"].get("versions", [V_NEW])) for h in hits]
        cids = [C[h]["metadata"].get("chunk_id", h) for h in hits]
        rows.append(dict(condition=cond, query_id=q["query_id"], template=q["template"], pair_class=q["pair_class"],
                         framing=q["framing"], has_evidence=bool(ev),
                         recall=(sum(e in hits for e in ev) / len(ev)) if ev else float("nan"),
                         n_old=sum(v == (V_OLD,) for v in vers),
                         n_new=sum(v == (V_NEW,) for v in vers) if cond == "rag_mixed" else float("nan"),
                         conflict=any(cids.count(x) > 1 for x in cids)))
RDF = pd.DataFrame(rows)
RDF.to_csv(OUT / "retrieval.csv", index=False)
cur = RDF[RDF.framing != "historical"]
tbl = cur.groupby(["condition", "pair_class"]).agg(n=("query_id", "count"), recall=("recall", "mean"),
                                                   n_old=("n_old", "mean"), conflict=("conflict", "mean")).round(3)
tbl

n  recall  n_old  conflict
condition pair_class                                  
rag_mixed ANCHOR_RENAMED   24   0.631  3.292     0.917
          ANSWER_CHANGED  144   0.958  1.743     0.993
          LABEL_CHANGED   160   0.956  1.431     0.981
          NEW_ONLY        110   0.802  0.791     0.227
          REVOKED_ANCHOR  112   0.493  3.071     0.143
          STABLE          320   0.793  1.091     0.659
rag_new   ANCHOR_RENAMED   24   0.761  0.000     0.000
          ANSWER_CHANGED  144   0.979  0.000     0.000
          LABEL_CHANGED   160   0.969  0.000     0.000
          NEW_ONLY        110   0.814  0.000     0.000
          REVOKED_ANCHOR  112   0.711  0.000     0.000
          STABLE          320   0.835  0.000     0.000

## 5. Dựng prompt

Mọi điều kiện dùng **cùng một lời dặn hệ thống** và **cùng câu hỏi**. Khác biệt duy nhất là có hay không có ngữ cảnh RAG, nên mọi chênh lệch kết quả đều quy được về ngữ cảnh.

- **Lời dặn hệ thống:** trợ lý CTI chuyên MITRE ATT&CK Enterprise, chỉ trả lời bằng JSON `{"answer": [...]}`, không biết thì để danh sách rỗng.
- **Yêu cầu theo kiểu đáp án:**
  - tactic: ghi mã **kèm tên**, ví dụ "TA0001 Initial Access". Tên được giữ lại để đo việc dùng nhãn lỗi thời ("Defense Evasion").
  - kỹ thuật / mitigation: chỉ ghi mã.
- **Ép định dạng (structured outputs):** vLLM chỉ cho phép mô hình sinh chuỗi đúng lược đồ JSON, ở dạng gọn (không có khoảng trắng thừa, nên câu trả lời không bị cắt giữa chừng). Mỗi phần tử phải đúng mẫu:
  - kỹ thuật: `T1234` / `T1234.001`;
  - mitigation: `M1234`;
  - tactic: chuỗi chữ–số tối đa 60 ký tự.

  Lược đồ đã được kiểm tra trên đúng thư viện `xgrammar 0.2.7` mà vLLM 0.31 dùng.
- **Ngữ cảnh RAG:** top-5 đoạn, mỗi đoạn tối đa 250 từ, giữ thứ tự xếp hạng. Văn bản **không** gắn nhãn phiên bản, giống một hệ RAG thông thường.

In [5]:
SYSTEM = ("You are a cyber threat intelligence (CTI) assistant with expertise in the MITRE ATT&CK Enterprise framework. "
          "Answer the user's question accurately. Respond only with a JSON object of the form {\"answer\": [...]}. "
          "Put only the requested items in the list. Use an empty list if you do not know.")
INSTR = {
    "tactic_set":     'List each tactic as its ATT&CK tactic ID followed by its name, for example "TA0001 Initial Access".',
    "technique_set":  "List ATT&CK technique IDs only (format T1234 or T1234.001).",
    "mitigation_set": "List ATT&CK mitigation IDs only (format M1234).",
}
PATTERN = {"technique_set": "^T[0-9]{4}(\\.[0-9]{3})?$", "mitigation_set": "^M[0-9]{4}$",
           "tactic_set": "^[A-Za-z0-9 ,.()&/'-]{1,60}$"}       # mã + tên tactic; không ký tự điều khiển

def schema_for(answer_type):
    item = {"type": "string", "pattern": PATTERN[answer_type]}   # không dùng maxLength cùng pattern (xgrammar bỏ qua)
    return {"type": "object",
            "properties": {"answer": {"type": "array", "items": item, "maxItems": 8}},
            "required": ["answer"], "additionalProperties": False}

def clip(text, n):
    w = text.split()
    return text if len(w) <= n else " ".join(w[:n]) + " …"

def build_messages(q, cond):
    if cond == "no_rag":
        user = f"Question: {q['question']}\n{INSTR[q['answer_type']]}"
    else:
        C = CORPORA[cond]
        ctx = "\n\n".join(f"[{k + 1}] {clip(C[h]['text'], MAX_CHUNK_WORDS)}" for k, h in enumerate(RETR[(cond, q['query_id'])]))
        user = ("Use the following context passages from a CTI knowledge base if they are relevant.\n\n"
                f"{ctx}\n\nQuestion: {q['question']}\n{INSTR[q['answer_type']]}")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

PROMPT_STATS = {}
for cond in CONDITIONS:
    rows = [dict(query_id=q["query_id"], condition=cond, answer_type=q["answer_type"],
                 messages=build_messages(q, cond), schema=schema_for(q["answer_type"]),
                 retrieved=RETR.get((cond, q["query_id"]), []))
            for q in QUERIES]
    with open(PROMPTS / f"{cond}.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n")
    words = [sum(len(m["content"].split()) for m in r["messages"]) for r in rows]
    PROMPT_STATS[cond] = dict(n=len(rows), words_median=int(pd.Series(words).median()), words_max=max(words))
MANIFEST["prompts"] = {c: sha256(PROMPTS / f"{c}.jsonl") for c in CONDITIONS}

# Bộ chạy thử: SMOKE_PER_TPL truy vấn đầu tiên của MỖI mẫu câu hỏi -> thử đủ 4 kiểu lược đồ JSON
smoke_ids = []
for tpl in sorted({q["template"] for q in QUERIES}):
    smoke_ids += [q["query_id"] for q in QUERIES if q["template"] == tpl][:SMOKE_PER_TPL]
smoke_set = set(smoke_ids)
for cond in CONDITIONS:
    with open(PROMPTS / f"{cond}.jsonl", encoding="utf-8") as fi, open(SMOKE_PROMPTS / f"{cond}.jsonl", "w", encoding="utf-8") as fo:
        for line in fi:
            if json.loads(line)["query_id"] in smoke_set:
                fo.write(line)
print(f"Bộ chạy thử: {len(smoke_ids)} truy vấn/điều kiện ({SMOKE_PER_TPL} mỗi mẫu câu hỏi)")
print(pd.DataFrame(PROMPT_STATS).T)
print("\nVí dụ prompt (rag_mixed):\n" + "\n".join(m["content"][:1200] for m in build_messages(QBYID["D1-T1222|current"], "rag_mixed")))

Bộ chạy thử: 8 truy vấn/điều kiện (2 mỗi mẫu câu hỏi)
              n  words_median  words_max
no_rag     1250            83        103
rag_new    1250           978       1333
rag_mixed  1250           982       1370

Ví dụ prompt (rag_mixed):
You are a cyber threat intelligence (CTI) assistant with expertise in the MITRE ATT&CK Enterprise framework. Answer the user's question accurately. Respond only with a JSON object of the form {"answer": [...]}. Put only the requested items in the list. Use an empty list if you do not know.
Use the following context passages from a CTI knowledge base if they are relevant.

[1] ATT&CK Technique T1222.002: Linux and Mac File and Directory Permissions Modification
Tactics: Defense Evasion (TA0005).
Sub-technique of: T1222 File and Directory Permissions Modification.
Platforms: macOS, Linux.
Adversaries may modify file or directory permissions/attributes to evade access control lists (ACLs) and access protected files. File and directory permissions a

## 6. Module suy luận `ctidrift_infer.py`

Mỗi mô hình chạy trong **một tiến trình Python riêng**. Khi tiến trình kết thúc, bộ nhớ GPU được giải phóng hoàn toàn, nên mô hình thứ hai không bị thiếu bộ nhớ (vLLM giữ GPU rất "chặt" trong cùng một tiến trình).

Module làm các việc sau:

1. Tải đúng phiên bản mô hình đã ghim (`snapshot_download` với mã commit) vào `/data/hf`.
2. **Kiểm tra độ dài** mọi prompt bằng chính tokenizer của mô hình **trước khi chạy**. Nếu prompt nào quá dài, nó dừng và báo, không để vLLM lỗi giữa chừng.
3. Chạy `llm.chat(...)` theo lô cho từng điều kiện, giải mã tham lam, ép JSON.
4. Ghi kết quả ra file tạm rồi đổi tên. File chỉ "xuất hiện" khi đã ghi xong, nên không bao giờ đọc phải file dở dang. Mỗi lần chạy có một **khoá chạy** (mã băm của file prompt, mô hình, phiên bản và tham số sinh). Chạy lại sẽ **chỉ bỏ qua phần đã xong với đúng khoá đó**. Nếu em đổi cấu hình (ví dụ `TOP_K`), phần liên quan tự chạy lại, không dùng nhầm kết quả cũ.
5. Ghi `meta_<điều kiện>.json`: phiên bản vLLM, torch, GPU, mã commit mô hình, thời gian chạy.

In [6]:
%%writefile ctidrift_infer.py
# Chạy suy luận cho MỘT mô hình trên các điều kiện của pilot (notebook 04). Gọi từ notebook qua subprocess.
import argparse, hashlib, json, os, sys, time, platform
from pathlib import Path

os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")


def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f]


def done_ok(path, meta_path, ids, run_key):
    # Kết quả cũ chỉ được dùng lại khi: đủ đúng các query_id theo đúng thứ tự VÀ cùng "khoá chạy"
    # (cùng file prompt, cùng mô hình/phiên bản, cùng tham số sinh). Đổi bất cứ thứ gì -> chạy lại.
    if not path.exists() or not meta_path.exists():
        return False
    try:
        got = [r["query_id"] for r in read_jsonl(path)]
        key = json.loads(meta_path.read_text()).get("run_key")
    except (ValueError, KeyError):
        return False
    return got == ids and key == run_key


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--revision", required=True)
    ap.add_argument("--tag", required=True)
    ap.add_argument("--prompts-dir", required=True)
    ap.add_argument("--conditions", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--max-model-len", type=int, default=6144)
    ap.add_argument("--max-tokens", type=int, default=96)
    ap.add_argument("--gpu-mem", type=float, default=0.85)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--chat-template-kwargs", default="{}")
    a = ap.parse_args()

    conds = [c for c in a.conditions.split(",") if c]
    out_dir = Path(a.out_dir) / a.tag
    out_dir.mkdir(parents=True, exist_ok=True)
    ctk = json.loads(a.chat_template_kwargs)

    from importlib.metadata import version as pkg_version
    script_sha = sha256_file(Path(__file__))
    vllm_ver = pkg_version("vllm")
    jobs, keys = {}, {}
    for c in conds:
        pfile = Path(a.prompts_dir) / f"{c}.jsonl"
        rows = read_jsonl(pfile)
        if a.limit:
            rows = rows[:a.limit]
        ids = [r["query_id"] for r in rows]
        keys[c] = hashlib.sha256(json.dumps(dict(prompts=sha256_file(pfile), limit=a.limit, repo=a.repo,
                                                 revision=a.revision, max_model_len=a.max_model_len,
                                                 max_tokens=a.max_tokens, seed=a.seed, ctk=ctk, gpu_mem=a.gpu_mem,
                                                 script=script_sha, vllm=vllm_ver),
                                            sort_keys=True).encode()).hexdigest()
        if done_ok(out_dir / f"{c}.jsonl", out_dir / f"meta_{c}.json", ids, keys[c]):
            print(f"[{a.tag}] {c}: đã có đủ {len(ids)} kết quả cùng cấu hình — bỏ qua", flush=True)
            continue
        jobs[c] = rows
    if not jobs:
        print(f"[{a.tag}] không còn việc gì để chạy", flush=True)
        return

    from huggingface_hub import snapshot_download
    path = snapshot_download(a.repo, revision=a.revision)
    print(f"[{a.tag}] mô hình tại {path}", flush=True)

    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path)
    budget = a.max_model_len - a.max_tokens
    for c, rows in jobs.items():
        lens = []
        for r in rows:
            text = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True, **ctk)
            lens.append(len(tok(text, add_special_tokens=False)["input_ids"]))
        too_long = [(r["query_id"], n) for r, n in zip(rows, lens) if n > budget]
        print(f"[{a.tag}] {c}: {len(rows)} prompt, dài nhất {max(lens)} token (giới hạn {budget})", flush=True)
        if too_long:
            sys.exit(f"LỖI: {len(too_long)} prompt vượt {budget} token, ví dụ {too_long[:3]} — tăng MAX_MODEL_LEN "
                     "hoặc giảm MAX_CHUNK_WORDS trong notebook")

    import torch
    import vllm
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import StructuredOutputsParams

    # disable_any_whitespace: JSON dạng gọn, mô hình không thể sinh khoảng trắng vô hạn rồi bị cắt ở max_tokens
    llm = LLM(model=path, tokenizer=path, max_model_len=a.max_model_len,
              gpu_memory_utilization=a.gpu_mem, seed=a.seed,
              structured_outputs_config={"backend": "xgrammar", "disable_any_whitespace": True})
    meta_base = dict(model_tag=a.tag, repo=a.repo, revision=a.revision, snapshot=Path(path).name,
                     vllm=vllm.__version__, torch=torch.__version__, cuda=torch.version.cuda,
                     gpu=torch.cuda.get_device_name(0), python=platform.python_version(),
                     max_model_len=a.max_model_len, max_tokens=a.max_tokens, gpu_mem=a.gpu_mem, seed=a.seed,
                     chat_template_kwargs=ctk, flashinfer_sampler=os.environ.get("VLLM_USE_FLASHINFER_SAMPLER"),
                     script_sha256=script_sha, structured_outputs="xgrammar, disable_any_whitespace")

    for c, rows in jobs.items():
        sps = [SamplingParams(temperature=0.0, max_tokens=a.max_tokens, seed=a.seed,
                              structured_outputs=StructuredOutputsParams(json=r["schema"])) for r in rows]
        t0 = time.time()
        outs = llm.chat([r["messages"] for r in rows], sps, use_tqdm=True, chat_template_kwargs=ctk or None)
        dt = time.time() - t0
        assert len(outs) == len(rows)
        tmp = out_dir / f".{c}.jsonl.tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            for r, o in zip(rows, outs):
                f.write(json.dumps(dict(query_id=r["query_id"], condition=c, model_tag=a.tag,
                                        text=o.outputs[0].text, finish_reason=o.outputs[0].finish_reason,
                                        n_prompt_tokens=len(o.prompt_token_ids or []),
                                        n_output_tokens=len(o.outputs[0].token_ids)), ensure_ascii=False) + "\n")
        os.replace(tmp, out_dir / f"{c}.jsonl")
        (out_dir / f"meta_{c}.json").write_text(json.dumps({**meta_base, "condition": c, "n": len(rows),
                                                            "run_key": keys[c], "seconds": round(dt, 1)}, indent=2))
        print(f"[{a.tag}] {c}: xong {len(rows)} truy vấn trong {dt / 60:.1f} phút", flush=True)


if __name__ == "__main__":
    main()

Writing ctidrift_infer.py


In [7]:
import signal

def run_infer(model, out_dir, prompts_dir, smoke=False):
    log = LOGS / f"{model['tag']}{'_smoke' if smoke else ''}.log"
    cmd = [sys.executable, "ctidrift_infer.py", "--repo", model["repo"], "--revision", model["revision"],
           "--tag", model["tag"], "--prompts-dir", str(prompts_dir), "--conditions", ",".join(CONDITIONS),
           "--out-dir", str(out_dir), "--max-model-len", str(MAX_MODEL_LEN), "--max-tokens", str(MAX_TOKENS),
           "--gpu-mem", str(GPU_MEM), "--seed", str(SEED),
           "--chat-template-kwargs", json.dumps(model["chat_template_kwargs"])]
    print(f"▶ {model['tag']}: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:\n    tail -f {log}")
    t0 = time.time()
    with open(log, "w", encoding="utf-8") as f:
        # start_new_session: tiến trình con có nhóm riêng; nếu em bấm dừng ô này, cả nhóm (gồm tiến trình
        # EngineCore của vLLM đang giữ GPU) bị tắt theo, không để lại tiến trình "mồ côi" chiếm bộ nhớ GPU
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, env=ENV, cwd=ROOT, start_new_session=True)
        try:
            p.wait()
        except BaseException:
            def _kill(sig):
                try:
                    os.killpg(p.pid, sig)
                except ProcessLookupError:
                    pass
            _kill(signal.SIGTERM)
            try:
                p.wait(timeout=60)
            except BaseException:
                pass
            _kill(signal.SIGKILL)                # luôn dọn sạch cả nhóm, kể cả khi bấm dừng lần hai
            print("⏹ Đã dừng và giải phóng GPU. Kiểm tra lại bằng lệnh nvidia-smi nếu cần.")
            raise
    tail = log.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(l for l in tail if l.startswith(f"[{model['tag']}]") or "LỖI" in l or "Error" in l)[-3000:])
    print(f"◀ {model['tag']}: {'XONG' if p.returncode == 0 else 'LỖI'} sau {(time.time() - t0) / 60:.1f} phút")
    if p.returncode != 0:
        print("\n".join(tail[-40:]))
        raise RuntimeError(f"Suy luận lỗi — gửi thầy file {log}")

## 7. Chạy thử nhanh rồi chạy toàn bộ

**7.1. Chạy thử (smoke test):** mỗi mô hình trả lời 8 truy vấn của mỗi điều kiện, gồm 2 truy vấn cho mỗi mẫu câu hỏi, để thử đủ cả 4 kiểu lược đồ JSON. Mục đích là phát hiện sớm lỗi cài đặt hoặc định dạng, trước khi chạy cả tiếng đồng hồ. Kết quả ghi vào thư mục riêng `raw_smoke/`. Ô sau kiểm tra **mọi câu trả lời đều là JSON hợp lệ**. Nếu không, notebook dừng.

In [8]:
SMOKE = OUT / "raw_smoke"
for m in MODELS:
    run_infer(m, SMOKE, SMOKE_PROMPTS, smoke=True)

smoke_rows = []
for m in MODELS:
    for cond in CONDITIONS:
        for r in map(json.loads, open(SMOKE / m["tag"] / f"{cond}.jsonl", encoding="utf-8")):
            q = QBYID[r["query_id"]]
            s = score_drift(q, r["text"], TACTIC_ALIAS)
            smoke_rows.append(dict(model=m["tag"], condition=cond, query_id=r["query_id"], text=r["text"][:80],
                                   format=s["format"], outcome=s["outcome"], gold=",".join(q["gold"])))
SM = pd.DataFrame(smoke_rows)
bad = SM[SM.format != "json"]
assert bad.empty, f"Có câu trả lời không phải JSON — gửi thầy bảng này:\n{bad.to_string()}"
print(f"✓ Chạy thử: {len(SM)} câu trả lời, 100% đúng định dạng JSON")
SM.head(12)

▶ qwen3-8b: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/pilot-18.1-19.2/logs/qwen3-8b_smoke.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] no_rag: 8 prompt, dài nhất 166 token (giới hạn 5984)
[qwen3-8b] rag_new: 8 prompt, dài nhất 1850 token (giới hạn 5984)
[qwen3-8b] rag_mixed: 8 prompt, dài nhất 1850 token (giới hạn 5984)
[qwen3-8b] no_rag: xong 8 truy vấn trong 0.0 phút
[qwen3-8b] rag_new: xong 8 truy vấn trong 0.0 phút
[qwen3-8b] rag_mixed: xong 8 truy vấn trong 0.0 phút
◀ qwen3-8b: XONG sau 4.8 phút
▶ llama3.1-8b: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/pilot-18.1-19.2/logs/llama3.1-8b_smoke.log
[llama3.1-8b] mô hình tại /data/hf/hub/models--hugging-quants--Meta-Llama-3.1-8B-Instruct-AWQ-INT4/snapshots/db1f81ad4b8c7e39777509fac66c652eb0a52f91
[llama3.1-8b] no_rag: 8 prompt, dài nhất 177 token (giới hạn 5984)
[

,model,condition,query_id,text,format,outcome,gold
0,qwen3-8b,no_rag,D1-T1011.001|unversioned,"{""answer"": [""TA0002 Initial Access"", ""TA0005 L...",json,PARTIAL,TA0010
1,qwen3-8b,no_rag,D1-T1011.001|current,"{""answer"": [""TA0006 Exfiltration""]}",json,WRONG,TA0010
2,qwen3-8b,no_rag,D2-T1001.001|unversioned,"{""answer"": []}",json,NO_ANSWER,T1001
3,qwen3-8b,no_rag,D2-T1001.001|current,"{""answer"": [""T1562""]}",json,WRONG,T1001
4,qwen3-8b,no_rag,D3-T1001|unversioned,"{""answer"": [""M1001""]}",json,WRONG,M1031
5,qwen3-8b,no_rag,D3-T1001|current,"{""answer"": [""M1001""]}",json,WRONG,M1031
6,qwen3-8b,no_rag,D4-T1001.001|unversioned,"{""answer"": [""T1001.001""]}",json,CORRECT,T1001.001
7,qwen3-8b,no_rag,D4-T1001.001|current,"{""answer"": [""T1001.001""]}",json,CORRECT,T1001.001
8,qwen3-8b,rag_new,D1-T1011.001|unversioned,"{""answer"": [""TA0010 Exfiltration""]}",json,CORRECT,TA0010
9,qwen3-8b,rag_new,D1-T1011.001|current,"{""answer"": [""TA0010 Exfiltration""]}",json,CORRECT,TA0010


**7.2. Chạy toàn bộ:** 1.250 truy vấn × 3 điều kiện × 2 mô hình = **7.500 lượt trả lời**. Ước tính 30–60 phút cho mỗi mô hình trên RTX 4070. Lần đầu còn phải tải mô hình, khoảng 5–6 GB mỗi mô hình.

Nếu em đóng notebook giữa chừng, chỉ cần **chạy lại từ đầu**: các điều kiện đã xong sẽ được bỏ qua.

In [9]:
for m in MODELS:
    run_infer(m, RAW, PROMPTS)

▶ qwen3-8b: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/pilot-18.1-19.2/logs/qwen3-8b.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] no_rag: 1250 prompt, dài nhất 173 token (giới hạn 5984)
[qwen3-8b] rag_new: 1250 prompt, dài nhất 2278 token (giới hạn 5984)
[qwen3-8b] rag_mixed: 1250 prompt, dài nhất 2282 token (giới hạn 5984)
[qwen3-8b] no_rag: xong 1250 truy vấn trong 0.5 phút
[qwen3-8b] rag_new: xong 1250 truy vấn trong 6.1 phút
[qwen3-8b] rag_mixed: xong 1250 truy vấn trong 5.6 phút
◀ qwen3-8b: XONG sau 12.8 phút
▶ llama3.1-8b: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/pilot-18.1-19.2/logs/llama3.1-8b.log
[llama3.1-8b] mô hình tại /data/hf/hub/models--hugging-quants--Meta-Llama-3.1-8B-Instruct-AWQ-INT4/snapshots/db1f81ad4b8c7e39777509fac66c652eb0a52f91
[llama3.1-8b] no_rag: 1250 prompt, dài nhất 184 token (giới h

## 8. Chấm điểm

Mỗi câu trả lời được chấm bằng `score_drift` (notebook 03):

- `outcome`: một trong `CORRECT / STALE / ANACHRONISM / PARTIAL / WRONG / NO_ANSWER`.
- `label_drift`: đúng mã nhưng dùng tên tactic của phiên bản kia.
- `format`: phải là `json` (nhờ ép định dạng).

Notebook này thêm ba cột phân tích:

- `changed`: đáp án **khác nhau** giữa hai phiên bản (`answer_changed` của notebook 03). Đây là nhóm "đáp án đã đổi" dùng trong mọi phân tích. Nó rộng hơn `drift_possible`, vì có cả trường hợp kỹ thuật chỉ được **thêm** tactic (ví dụ T1127: cũ {TA0005}, mới {TA0002, TA0005}).
- `other_exact`: câu trả lời **trùng khớp hoàn toàn đáp án của phiên bản kia**. Với T1127, trả lời đúng tập cũ {TA0005} thì `score_drift` xếp là `PARTIAL`, vì không có phần tử nào *chỉ* đúng ở bản cũ, nhưng thực chất đó vẫn là câu trả lời lỗi thời.
- `drift_ext` = `outcome` là `STALE` / `ANACHRONISM` **hoặc** `other_exact`. Đây là **chỉ số lỗi trôi chính** của notebook này: với câu hỏi về bản hiện hành nó là "lỗi thời", với câu hỏi về bản cũ nó là "nhảy sang bản mới".
- `subtype` (chỉ câu D1 có đáp án đổi, hỏi về bản hiện hành): `NEW_TACTIC` nếu đáp án mới chứa tactic **chưa từng có** ở bản cũ (TA0112), `EXISTING_TACTICS` nếu chỉ chuyển giữa các tactic đã có.
- `recall`: tỉ lệ bằng chứng được truy hồi (mục 4), dùng để tách trường hợp "RAG đã đưa đúng bằng chứng mà mô hình vẫn sai".

Ta kiểm tra thêm: mỗi (mô hình, điều kiện) có **đủ 1.250 câu trả lời**, và đếm số câu bị cắt do hết `MAX_TOKENS` (`finish_reason = length`).

In [10]:
import numpy as np
_alias = json.loads(REQUIRED["tactic_alias"].read_text())
OLD_TACTIC_IDS = {t for n, t in _alias["alias"].items() if n not in _alias["new_only_labels"]}
NEW_TACTIC_IDS = set(_alias["alias"].values()) - OLD_TACTIC_IDS
print("Tactic chỉ có ở bản mới:", NEW_TACTIC_IDS)

RECALL = {(r.condition, r.query_id): r.recall for r in RDF.itertuples()}

def subtype(q):
    if q["template"] != "D1_tech_tactic" or not q["answer_changed"] or q["direction"] != "current":
        return ""
    return "NEW_TACTIC" if (set(q["gold"]) - set(q["other_gold"])) & NEW_TACTIC_IDS else "EXISTING_TACTICS"

rows = []
for m in MODELS:
    for cond in CONDITIONS:
        p = RAW / m["tag"] / f"{cond}.jsonl"
        got = [json.loads(l) for l in open(p, encoding="utf-8")]
        assert [r["query_id"] for r in got] == [q["query_id"] for q in QUERIES], f"{p}: thiếu hoặc sai thứ tự truy vấn"
        for r in got:
            q = QBYID[r["query_id"]]
            s = score_drift(q, r["text"], TACTIC_ALIAS)
            pred, gold, other = set(s["pred"]), set(q["gold"]), set(q["other_gold"])
            other_exact = bool(pred) and pred == other and other != gold
            rows.append(dict(model=m["tag"], condition=cond, query_id=q["query_id"], item_id=q["item_id"],
                             template=q["template"], pair_class=q["pair_class"], framing=q["framing"],
                             direction=q["direction"], changed=q["answer_changed"], drift_possible=q["drift_possible"],
                             split=q["split"],
                             change_types="|".join(q["change_types"]), subtype=subtype(q),
                             recall=RECALL.get((cond, q["query_id"]), float("nan")), answer=r["text"],
                             finish_reason=r["finish_reason"], n_prompt_tokens=r["n_prompt_tokens"],
                             pred="|".join(s["pred"]), gold="|".join(q["gold"]), other_gold="|".join(q["other_gold"]),
                             em=s["em"], f1=s["f1"], outcome=s["outcome"], stale=s["outcome"] == "STALE",
                             other_exact=other_exact,
                             drift_ext=s["outcome"] in ("STALE", "ANACHRONISM") or other_exact,
                             label_drift=s["label_drift"], format=s["format"]))
SC = pd.DataFrame(rows)
SC.to_csv(OUT / "scored.csv", index=False)
chk = SC.groupby(["model", "condition"]).agg(n=("query_id", "count"), json=("format", lambda x: (x == "json").mean()),
                                             truncated=("finish_reason", lambda x: (x == "length").sum()),
                                             prompt_tokens_max=("n_prompt_tokens", "max"))
assert (chk.n == len(QUERIES)).all()
chk

Tactic chỉ có ở bản mới: {'TA0112'}


n  json  truncated  prompt_tokens_max
model       condition                                          
llama3.1-8b no_rag     1250   1.0          0                184
            rag_mixed  1250   1.0          0               2097
            rag_new    1250   1.0          0               2126
qwen3-8b    no_rag     1250   1.0          0                173
            rag_mixed  1250   1.0          0               2282
            rag_new    1250   1.0          0               2278

## 9. Kết quả

**Ký hiệu:**

- **acc**: tỉ lệ trả lời đúng hoàn toàn (`em`).
- **drift_ext** (chỉ số chính): tỉ lệ trả lời theo **phiên bản kia**, tính trên truy vấn có đáp án đổi (`changed`). Với câu hỏi về bản hiện hành, đây là tỉ lệ **lỗi thời**.
- **stale (chặt)**: định nghĩa của `score_drift` (có ít nhất một mã chỉ đúng ở bản cũ). Chỉ tính trên `drift_possible`, để tham khảo.
- **[a, b]**: khoảng tin cậy 95% **bootstrap theo cụm mục** (`item_id`, 2.000 lần lặp). Khoảng này tính đến việc một mục sinh nhiều truy vấn, nên rộng hơn và trung thực hơn khoảng coi từng truy vấn là độc lập.

Ô dưới định nghĩa các hàm thống kê. Wilson và Fisher dùng để tham khảo; bootstrap theo cụm dùng cho tiêu chí đi tiếp / dừng.

In [11]:
def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def fisher_two_sided(a, b, c, d):
    # Kiểm định chính xác Fisher, hai phía, cho bảng 2x2 [[a, b], [c, d]].
    # Tính trong không gian log (lgamma) để không tràn số khi n lớn; khớp scipy.stats.fisher_exact.
    n1, n2, k = a + b, c + d, a + c
    def logc(n, r):
        return math.lgamma(n + 1) - math.lgamma(r + 1) - math.lgamma(n - r + 1)
    def pmf(x):
        return math.exp(logc(n1, x) + logc(n2, k - x) - logc(n1 + n2, k))
    p_obs = pmf(a)
    lo, hi = max(0, k - n2), min(k, n1)
    return min(1.0, sum(pmf(x) for x in range(lo, hi + 1) if pmf(x) <= p_obs * (1 + 1e-7)))

def _agg(df, col):
    g = df.groupby("item_id")[col].agg(["sum", "count"])
    return g["sum"].to_numpy(float), g["count"].to_numpy(float), g.index

def boot_rate(df, col):
    # Tỉ lệ trung bình của cột col và khoảng tin cậy 95% bootstrap theo cụm item_id
    if len(df) == 0:
        return (float("nan"),) * 3
    s, c, _ = _agg(df, col)
    idx = np.random.default_rng(SEED).integers(0, len(s), size=(BOOT_B, len(s)))
    r = s[idx].sum(1) / c[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (s.sum() / c.sum(), lo, hi)

def boot_diff_paired(df_a, df_b, col):
    # Hiệu tỉ lệ (a − b) trên CÙNG các mục (ví dụ cùng truy vấn ở hai điều kiện); bootstrap theo cụm mục
    sa, ca, ia = _agg(df_a, col)
    sb, cb, ib = _agg(df_b, col)
    assert list(ia) == list(ib), "hai nhóm phải có cùng tập mục"
    if len(sa) == 0:
        return (float("nan"),) * 3
    idx = np.random.default_rng(SEED).integers(0, len(sa), size=(BOOT_B, len(sa)))
    r = sa[idx].sum(1) / ca[idx].sum(1) - sb[idx].sum(1) / cb[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi)

def boot_diff_indep(df_a, df_b, col):
    # Hiệu tỉ lệ (a − b) giữa hai nhóm mục KHÁC nhau; mỗi nhóm được lấy mẫu lại độc lập
    sa, ca, _ = _agg(df_a, col)
    sb, cb, _ = _agg(df_b, col)
    if len(sa) == 0 or len(sb) == 0:
        return (float("nan"),) * 3
    rng = np.random.default_rng(SEED)
    ia = rng.integers(0, len(sa), size=(BOOT_B, len(sa)))
    ib = rng.integers(0, len(sb), size=(BOOT_B, len(sb)))
    r = sa[ia].sum(1) / ca[ia].sum(1) - sb[ib].sum(1) / cb[ib].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi)

def fmt_b(t, n=None):
    v, lo, hi = t
    if v != v:
        return "—"
    return f"{v:.2f} [{lo:.2f}, {hi:.2f}]" + (f" (n={n})" if n is not None else "")

CUR = SC[SC.direction == "current"]
HIS = SC[SC.direction == "historical"]
print("Sẵn sàng:", len(CUR), "truy vấn về bản hiện hành,", len(HIS), "truy vấn về bản cũ")

Sẵn sàng: 5220 truy vấn về bản hiện hành, 2280 truy vấn về bản cũ


### 9.1. Bảng tổng: đúng / lỗi thời theo mô hình × điều kiện

Chỉ tính câu hỏi về **bản hiện hành** (`unversioned` + `current`). Hai nhóm được so sánh:

- `STABLE`: đáp án không đổi giữa hai phiên bản (đối chứng).
- `changed`: đáp án đã đổi, tức trả lời theo bản cũ là sai.

Cột `p Fisher` coi từng truy vấn là độc lập, chỉ để tham khảo.

In [12]:
T1 = []
for (m, c), g in CUR.groupby(["model", "condition"], sort=False):
    s, d = g[g.pair_class == "STABLE"], g[g.changed]
    T1.append({"model": m, "condition": c,
               "acc STABLE": fmt_b(boot_rate(s, "em"), len(s)),
               "acc changed": fmt_b(boot_rate(d, "em"), len(d)),
               "drift_ext changed": fmt_b(boot_rate(d, "drift_ext")),
               "stale (chặt)": fmt_b(boot_rate(d[d.drift_possible], "stale")),
               "p Fisher (acc)": f"{fisher_two_sided(int(s.em.sum()), len(s) - int(s.em.sum()), int(d.em.sum()), len(d) - int(d.em.sum())):.1e}"})
T1 = pd.DataFrame(T1)
T1.to_csv(OUT / "summary_overall.csv", index=False)
T1

,model,condition,acc STABLE,acc changed,drift_ext changed,stale (chặt),p Fisher (acc)
0,qwen3-8b,no_rag,"0.29 [0.22, 0.36] (n=320)","0.01 [0.00, 0.03] (n=224)","0.22 [0.16, 0.29]","0.23 [0.16, 0.30]",1.8e-21
1,qwen3-8b,rag_new,"0.88 [0.83, 0.92] (n=320)","0.84 [0.79, 0.90] (n=224)","0.04 [0.01, 0.08]","0.05 [0.01, 0.08]",2.6e-01
2,qwen3-8b,rag_mixed,"0.88 [0.83, 0.92] (n=320)","0.16 [0.10, 0.22] (n=224)","0.83 [0.75, 0.89]","0.86 [0.80, 0.92]",1.6e-68
3,llama3.1-8b,no_rag,"0.18 [0.13, 0.24] (n=320)","0.00 [0.00, 0.01] (n=224)","0.60 [0.51, 0.68]","0.60 [0.51, 0.68]",7.8e-14
4,llama3.1-8b,rag_new,"0.83 [0.78, 0.88] (n=320)","0.75 [0.69, 0.82] (n=224)","0.11 [0.07, 0.16]","0.11 [0.07, 0.16]",3.9e-02
5,llama3.1-8b,rag_mixed,"0.80 [0.75, 0.85] (n=320)","0.08 [0.04, 0.13] (n=224)","0.91 [0.85, 0.95]","0.94 [0.90, 0.98]",7.1e-68


**Lưu ý khi đọc bảng 9.1:** nhóm `changed` chủ yếu là câu D1 (tactic), còn nhóm `STABLE` chia đều 4 mẫu câu hỏi. Vì vậy bảng 9.2 và các tiêu chí ở mục 10 so sánh **trong cùng mẫu D1**.

### 9.2. Theo mẫu câu hỏi × lớp thay đổi (câu hỏi về bản hiện hành)

In [13]:
T2 = []
for (m, c, t, k), g in CUR.groupby(["model", "condition", "template", "pair_class"], sort=True):
    dp = g[g.changed]
    T2.append(dict(model=m, condition=c, template=t, pair_class=k, n=len(g), acc=g.em.mean(),
                   drift_ext=dp.drift_ext.mean() if len(dp) else float("nan"),
                   stale=g[g.drift_possible].stale.mean() if g.drift_possible.any() else float("nan"), n_changed=len(dp),
                   label_drift=g.label_drift.mean() if t == "D1_tech_tactic" else float("nan")))
T2 = pd.DataFrame(T2)
T2.to_csv(OUT / "summary_by_class.csv", index=False)
print("Tỉ lệ lỗi thời drift_ext (chỉ các lớp có đáp án đổi):")
T2.pivot_table(index=["template", "pair_class"], columns=["model", "condition"], values="drift_ext").round(2)

Tỉ lệ lỗi thời drift_ext (chỉ các lớp có đáp án đổi):


model                            llama3.1-8b                   qwen3-8b  \
condition                             no_rag rag_mixed rag_new   no_rag   
template          pair_class                                              
D1_tech_tactic    ANSWER_CHANGED        0.57      0.86    0.04     0.04   
                  REVOKED_ANCHOR        0.90      1.00    0.33     0.13   
D2_subtech_parent REVOKED_ANCHOR        0.00      0.94    0.00     0.38   
D4_id_currency    REVOKED_ANCHOR        0.74      1.00    0.24     1.00   

model                                               
condition                        rag_mixed rag_new  
template          pair_class                        
D1_tech_tactic    ANSWER_CHANGED      0.75    0.01  
                  REVOKED_ANCHOR      1.00    0.00  
D2_subtech_parent REVOKED_ANCHOR      0.94    0.00  
D4_id_currency    REVOKED_ANCHOR      0.94    0.24

In [14]:
print("Tỉ lệ trả lời đúng:")
T2.pivot_table(index=["template", "pair_class"], columns=["model", "condition"], values="acc").round(2)

Tỉ lệ trả lời đúng:


model                             llama3.1-8b                   qwen3-8b  \
condition                              no_rag rag_mixed rag_new   no_rag   
template           pair_class                                              
D1_tech_tactic     ANCHOR_RENAMED        0.00      1.00    1.00     0.00   
                   ANSWER_CHANGED        0.01      0.12    0.86     0.01   
                   LABEL_CHANGED         0.22      0.94    0.90     0.01   
                   NEW_ONLY              0.07      0.61    0.80     0.00   
                   REVOKED_ANCHOR        0.06      0.09    0.44     0.00   
                   STABLE                0.05      0.88    0.90     0.08   
D2_subtech_parent  ANCHOR_RENAMED        0.00      1.00    0.50     0.00   
                   NEW_ONLY              0.04      0.33    0.58     0.00   
                   REVOKED_ANCHOR        0.00      0.06    0.88     0.00   
                   STABLE                0.08      0.61    0.72     0.08   
D3_tech_mitigation ANCHOR_RENAMED        0.00      0.88    0.88     0.00   
                   NEW_ONLY              0.00      0.92    0.95     0.00   
                   REVOKED_ANCHOR        0.00      0.96    0.79     0.00   
                   STABLE                0.00      0.82    0.80     0.00   
D4_id_currency     ANCHOR_RENAMED        0.50      0.88    1.00     1.00   
                   REVOKED_ANCHOR        0.00      0.00    0.47     0.00   
                   STABLE                0.61      0.90    0.89     1.00   

model                                                
condition                         rag_mixed rag_new  
template           pair_class                        
D1_tech_tactic     ANCHOR_RENAMED      1.00    1.00  
                   ANSWER_CHANGED      0.23    0.92  
                   LABEL_CHANGED       0.98    0.98  
                   NEW_ONLY            0.91    1.00  
                   REVOKED_ANCHOR      0.12    0.65  
                   STABLE              0.96    0.96  
D2_subtech_parent  ANCHOR_RENAMED      1.00    1.00  
                   NEW_ONLY            0.58    0.75  
                   REVOKED_ANCHOR      0.06    1.00  
                   STABLE              0.76    0.85  
D3_tech_mitigation ANCHOR_RENAMED      1.00    0.62  
                   NEW_ONLY            0.82    0.80  
                   REVOKED_ANCHOR      0.89    0.54  
                   STABLE              0.79    0.70  
D4_id_currency     ANCHOR_RENAMED      1.00    1.00  
                   REVOKED_ANCHOR      0.03    0.62  
                   STABLE              1.00    1.00

### 9.3. Xung đột tri thức: RAG đã đưa đúng bằng chứng mà mô hình vẫn trả lời theo bản cũ?

Đây là bảng **quan trọng nhất** cho đề tài. Xét điều kiện `rag_new`, câu hỏi có đáp án đổi. Bảng chia truy vấn theo việc **bằng chứng của bản mới có được truy hồi đầy đủ không** (`recall = 1`):

- Nếu `recall = 1` mà mô hình vẫn trả lời theo bản cũ, mô hình đang **ưu tiên trí nhớ cũ hơn ngữ cảnh mới**. Đây là hiện tượng *knowledge conflict* (ClashEval, Xie et al. 2024), nay đo trong bối cảnh CTI.
- Nếu `recall < 1`, lỗi có thể đến từ **tầng truy hồi**.

In [15]:
RN = CUR[(CUR.condition == "rag_new") & CUR.changed]
T7 = []
for m, g in RN.groupby("model", sort=False):
    for lab, sub in (("recall = 1", g[g.recall == 1]), ("recall < 1", g[g.recall < 1]), ("không có bằng chứng", g[g.recall.isna()])):
        T7.append({"model": m, "nhóm": lab, "n": len(sub), "acc": fmt_b(boot_rate(sub, "em")),
                   "drift_ext": fmt_b(boot_rate(sub, "drift_ext")),
                   "stale (chặt)": fmt_b(boot_rate(sub[sub.drift_possible], "stale"))})
T7 = pd.DataFrame(T7)
T7.to_csv(OUT / "summary_conflict.csv", index=False)
T7

,model,nhóm,n,acc,drift_ext,stale (chặt)
0,qwen3-8b,recall = 1,180,"0.92 [0.88, 0.96]","0.01 [0.00, 0.02]","0.01 [0.00, 0.02]"
1,qwen3-8b,recall < 1,10,"0.20 [0.00, 0.44]","0.10 [0.00, 0.38]","0.10 [0.00, 0.38]"
2,qwen3-8b,không có bằng chứng,34,"0.62 [0.41, 0.82]","0.24 [0.06, 0.44]","0.24 [0.06, 0.44]"
3,llama3.1-8b,recall = 1,180,"0.84 [0.77, 0.90]","0.06 [0.02, 0.09]","0.06 [0.02, 0.10]"
4,llama3.1-8b,recall < 1,10,"0.20 [0.00, 0.44]","0.60 [0.25, 0.90]","0.60 [0.25, 0.90]"
5,llama3.1-8b,không có bằng chứng,34,"0.47 [0.26, 0.68]","0.24 [0.09, 0.41]","0.24 [0.09, 0.41]"


### 9.4. Kho trộn có làm tệ hơn không? (`rag_mixed` − `rag_new`, cùng truy vấn)

Hiệu số dương nghĩa là kho trộn làm **tăng** lỗi thời. Khoảng tin cậy là bootstrap **ghép cặp** (cùng truy vấn ở hai điều kiện, lấy mẫu lại theo mục).

In [16]:
T8 = []
for m in [x["tag"] for x in MODELS]:
    base = CUR[(CUR.model == m) & CUR.changed]
    for lab, sel in (("tất cả mẫu", base), ("D1–D3 (bỏ D4)", base[base.template != "D4_id_currency"])):
        a = sel[sel.condition == "rag_mixed"].sort_values("query_id")
        b = sel[sel.condition == "rag_new"].sort_values("query_id")
        T8.append({"model": m, "phạm vi": lab, "n": len(a),
                   "Δ drift_ext (mixed − new)": fmt_b(boot_diff_paired(a, b, "drift_ext")),
                   "Δ acc (mixed − new)": fmt_b(boot_diff_paired(a, b, "em"))})
T8 = pd.DataFrame(T8)
T8.to_csv(OUT / "summary_mixed_vs_new.csv", index=False)
T8

,model,phạm vi,n,Δ drift_ext (mixed − new),Δ acc (mixed − new)
0,qwen3-8b,tất cả mẫu,224,"0.78 [0.71, 0.85]","-0.69 [-0.76, -0.61]"
1,qwen3-8b,D1–D3 (bỏ D4),190,"0.79 [0.72, 0.87]","-0.71 [-0.78, -0.62]"
2,llama3.1-8b,tất cả mẫu,224,"0.80 [0.73, 0.86]","-0.67 [-0.74, -0.59]"
3,llama3.1-8b,D1–D3 (bỏ D4),190,"0.81 [0.74, 0.87]","-0.71 [-0.77, -0.63]"


### 9.5. Trong D1: tactic hoàn toàn mới (TA0112) so với chuyển giữa các tactic đã có

In [17]:
D1d = CUR[(CUR.template == "D1_tech_tactic") & (CUR.subtype != "")]
T9 = []
for (m, c), g in D1d.groupby(["model", "condition"], sort=False):
    a, b = g[g.subtype == "NEW_TACTIC"], g[g.subtype == "EXISTING_TACTICS"]
    T9.append({"model": m, "condition": c, "n NEW/EXIST": f"{len(a)}/{len(b)}",
               "drift_ext NEW_TACTIC": fmt_b(boot_rate(a, "drift_ext")),
               "drift_ext EXISTING": fmt_b(boot_rate(b, "drift_ext")),
               "Δ (NEW − EXIST)": fmt_b(boot_diff_indep(a, b, "drift_ext"))})
T9 = pd.DataFrame(T9)
T9.to_csv(OUT / "summary_d1_subtype.csv", index=False)
T9

,model,condition,n NEW/EXIST,drift_ext NEW_TACTIC,drift_ext EXISTING,Δ (NEW − EXIST)
0,qwen3-8b,no_rag,112/62,"0.08 [0.03, 0.14]","0.02 [0.00, 0.05]","0.06 [0.00, 0.13]"
1,qwen3-8b,rag_new,112/62,"0.01 [0.00, 0.03]","0.02 [0.00, 0.05]","-0.01 [-0.05, 0.03]"
2,qwen3-8b,rag_mixed,112/62,"0.98 [0.96, 1.00]","0.45 [0.29, 0.61]","0.53 [0.36, 0.69]"
3,llama3.1-8b,no_rag,112/62,"0.72 [0.61, 0.83]","0.45 [0.31, 0.60]","0.27 [0.09, 0.45]"
4,llama3.1-8b,rag_new,112/62,"0.13 [0.07, 0.21]","0.02 [0.00, 0.05]","0.12 [0.05, 0.20]"
5,llama3.1-8b,rag_mixed,112/62,"1.00 [1.00, 1.00]","0.68 [0.53, 0.81]","0.32 [0.18, 0.47]"


### 9.6. Hiện tượng "cùng mã, đổi nghĩa" (TA0005)

Với các kỹ thuật vẫn thuộc TA0005 (`LABEL_CHANGED`), mã đúng không đổi nhưng tên đã đổi từ *Defense Evasion* thành *Stealth*. Hai cột cuối cho biết tỉ lệ câu trả lời dùng **tên cũ**: trên mọi câu trả lời, và riêng trên các câu **đúng mã**.

In [18]:
lc = CUR[(CUR.template == "D1_tech_tactic") & (CUR.pair_class == "LABEL_CHANGED")]
T3 = pd.DataFrame([{"model": m, "condition": c, "acc (mã)": fmt_b(boot_rate(g, "em"), len(g)),
                    "dùng tên cũ (mọi câu)": fmt_b(boot_rate(g, "label_drift")),
                    "dùng tên cũ (câu đúng mã)": fmt_b(boot_rate(g[g.em], "label_drift"), int(g.em.sum()))}
                   for (m, c), g in lc.groupby(["model", "condition"], sort=False)])
T3.to_csv(OUT / "summary_label_drift.csv", index=False)
T3

,model,condition,acc (mã),dùng tên cũ (mọi câu),dùng tên cũ (câu đúng mã)
0,qwen3-8b,no_rag,"0.01 [0.00, 0.02] (n=160)","0.03 [0.01, 0.07]","1.00 [1.00, 1.00] (n=1)"
1,qwen3-8b,rag_new,"0.98 [0.96, 1.00] (n=160)","0.00 [0.00, 0.00]","0.00 [0.00, 0.00] (n=157)"
2,qwen3-8b,rag_mixed,"0.98 [0.96, 1.00] (n=160)","0.93 [0.89, 0.97]","0.95 [0.90, 0.98] (n=157)"
3,llama3.1-8b,no_rag,"0.23 [0.16, 0.29] (n=160)","0.20 [0.13, 0.28]","0.19 [0.06, 0.35] (n=36)"
4,llama3.1-8b,rag_new,"0.90 [0.84, 0.95] (n=160)","0.01 [0.00, 0.02]","0.00 [0.00, 0.00] (n=144)"
5,llama3.1-8b,rag_mixed,"0.94 [0.90, 0.98] (n=160)","0.86 [0.80, 0.92]","0.87 [0.80, 0.93] (n=151)"


### 9.7. Cách hỏi: `unversioned` so với `current`

Nhắc lại: `current` vừa nói rõ phiên bản **vừa** có câu gợi ý "nếu mã đã bị thay thế…". Chênh lệch giữa hai cột là tác động gộp của cả hai yếu tố.

In [19]:
T4 = (CUR[CUR.changed].pivot_table(index=["model", "condition"], columns="framing", values="drift_ext", aggfunc="mean").round(3))
T4

framing                current  unversioned
model       condition                      
llama3.1-8b no_rag       0.607        0.589
            rag_mixed    0.884        0.929
            rag_new      0.152        0.062
qwen3-8b    no_rag       0.214        0.232
            rag_mixed    0.804        0.848
            rag_new      0.062        0.027

### 9.8. Câu hỏi về bản cũ (`historical`): có bị "nhảy" sang bản mới không?

Ở đây `drift_ext` nghĩa là trả lời theo **bản mới** khi câu hỏi hỏi về v18.1 (`ANACHRONISM` hoặc trùng khớp đáp án bản mới).

**Lưu ý:** ở điều kiện `rag_new`, kho chỉ chứa nội dung v19.2, nên ngữ cảnh **không thể** hỗ trợ câu trả lời theo v18.1. Tỉ lệ "nhảy sang bản mới" ở đó một phần là do thiết kế. So sánh có ý nghĩa nhất là `no_rag` với `rag_mixed`. Hiện tượng này đáng chú ý chủ yếu ở điều kiện RAG, khi ngữ cảnh chứa nội dung bản mới.

In [20]:
HD = HIS[HIS.changed]
T5 = pd.DataFrame([{"model": m, "condition": c, "acc": fmt_b(boot_rate(g, "em"), len(g)),
                    "nhảy sang bản mới (drift_ext)": fmt_b(boot_rate(g, "drift_ext"))}
                   for (m, c), g in HD.groupby(["model", "condition"], sort=False)])
T5

,model,condition,acc,nhảy sang bản mới (drift_ext)
0,qwen3-8b,no_rag,"0.21 [0.14, 0.29] (n=112)","0.09 [0.04, 0.15]"
1,qwen3-8b,rag_new,"0.02 [0.00, 0.04] (n=112)","0.88 [0.82, 0.94]"
2,qwen3-8b,rag_mixed,"0.50 [0.41, 0.59] (n=112)","0.48 [0.39, 0.57]"
3,llama3.1-8b,no_rag,"0.19 [0.12, 0.27] (n=112)","0.00 [0.00, 0.00]"
4,llama3.1-8b,rag_new,"0.03 [0.00, 0.06] (n=112)","0.89 [0.83, 0.95]"
5,llama3.1-8b,rag_mixed,"0.42 [0.33, 0.51] (n=112)","0.54 [0.46, 0.63]"


### 9.9. Phân bố kết quả chi tiết (câu hỏi về bản hiện hành)

In [21]:
T6 = pd.crosstab([CUR.model, CUR.condition], CUR.outcome, normalize="index").round(3)
T6.to_csv(OUT / "summary_outcomes.csv")
T6

outcome                CORRECT  NO_ANSWER  PARTIAL  STALE  WRONG
model       condition                                           
llama3.1-8b no_rag       0.122      0.000    0.229  0.147  0.502
            rag_mixed    0.634      0.005    0.089  0.232  0.040
            rag_new      0.815      0.005    0.114  0.028  0.039
qwen3-8b    no_rag       0.118      0.018    0.041  0.057  0.764
            rag_mixed    0.707      0.029    0.031  0.213  0.021
            rag_new      0.876      0.033    0.057  0.011  0.022

## 10. Đánh giá đi tiếp hay dừng (go / no-go)

Các tiêu chí được thiết kế để **không thể đạt chỉ nhờ giới hạn tri thức của mô hình**. Riêng việc mô hình cũ không biết TA0112 thì đã được dự đoán trước, không tính là phát hiện. Mọi so sánh dùng khoảng tin cậy bootstrap theo cụm mục. Ngưỡng là **gợi ý**; quyết định cuối cùng thầy và em cùng đưa ra sau khi đọc mục 9.

| Tiêu chí | Vai trò | Điều kiện đạt (ở ít nhất 1 mô hình) |
|---|---|---|
| **G1 – Kiểm tra nền** | Quy trình đo hoạt động đúng | `no_rag`, mẫu D1: acc(STABLE) − acc(changed) có cận dưới khoảng tin cậy > 0,10 |
| **G2 – Phát hiện chính** | RAG **không** tự giải quyết được trôi phiên bản | (a) `rag_new`, đã truy hồi đủ bằng chứng (`recall = 1`): tỉ lệ `drift_ext` ≥ 0,10 với cận dưới ≥ 0,05; **hoặc** (b) `rag_mixed` − `rag_new` về `drift_ext` (ghép cặp): cận dưới > 0 |
| **G3 – Bổ sung** | Lỗi phụ thuộc kiểu thay đổi | Trong D1, ở một điều kiện nào đó: hiệu `drift_ext` giữa `NEW_TACTIC` và `EXISTING_TACTICS` có khoảng tin cậy không chứa 0 |

**Cách đọc:**

- G1 gần như chắc chắn đạt, vì mô hình phát hành trước v19 không thể biết TA0112. Nó chỉ xác nhận quy trình đo hoạt động. Vì vậy **kết luận thực chất nằm ở G2**.
- G2 được xét ở 2 mô hình × 2 điều kiện (a, b), tức có 4 cơ hội đạt. G3 lấy kết quả tốt nhất trong 3 điều kiện × 2 mô hình. Khi báo cáo phải nói rõ điều này, và G3 chỉ là **phân tích khám phá**.

**Kết luận:**

- **GO** khi đạt G1 và G2. G3 làm bài báo mạnh hơn nhưng không bắt buộc.
- **Nếu G1 không đạt:** quy trình có vấn đề (chấm điểm, prompt, mô hình), cần kiểm tra trước khi diễn giải bất cứ điều gì.
- **Nếu G1 đạt mà G2 không đạt:** RAG với kho bản mới chữa được hầu hết lỗi. Đề tài vẫn có giá trị (đo lường, thẻ lỗi thời ngoài thực tế ở Trụ 1), nhưng trọng tâm Trụ 2–3 phải điều chỉnh. Thầy sẽ bàn với em.

In [22]:
G = {"G1": [], "G2": [], "G3": []}
for m in [x["tag"] for x in MODELS]:
    d1 = CUR[(CUR.model == m) & (CUR.condition == "no_rag") & (CUR.template == "D1_tech_tactic")]
    dif = boot_diff_indep(d1[d1.pair_class == "STABLE"], d1[d1.changed], "em")
    G["G1"].append((m, dif[1] > 0.10, f"no_rag D1: acc STABLE − acc changed = {fmt_b(dif)}"))

    rn = CUR[(CUR.model == m) & (CUR.condition == "rag_new") & CUR.changed & (CUR.recall == 1)]
    a = boot_rate(rn, "drift_ext")
    ok_a = a[0] >= 0.10 and a[1] >= 0.05
    base = CUR[(CUR.model == m) & CUR.changed]
    b = boot_diff_paired(base[base.condition == "rag_mixed"].sort_values("query_id"),
                         base[base.condition == "rag_new"].sort_values("query_id"), "drift_ext")
    ok_b = b[1] > 0
    G["G2"].append((m, ok_a or ok_b, f"(a) rag_new recall=1: drift_ext = {fmt_b(a, len(rn))} {'✓' if ok_a else '·'} | "
                                     f"(b) mixed − new: Δdrift_ext = {fmt_b(b)} {'✓' if ok_b else '·'}"))

    best = None
    for c in CONDITIONS:
        g = CUR[(CUR.model == m) & (CUR.condition == c) & (CUR.subtype != "")]
        dd = boot_diff_indep(g[g.subtype == "NEW_TACTIC"], g[g.subtype == "EXISTING_TACTICS"], "drift_ext")
        sig = dd[1] > 0 or dd[2] < 0
        if best is None or (sig and not best[0]):
            best = (sig, f"{c}: Δ(NEW − EXIST) = {fmt_b(dd)}")
    G["G3"].append((m, best[0], best[1]))

res = {}
for k, items in G.items():
    res[k] = any(ok for _, ok, _ in items)
    print(f"{k}: {'ĐẠT' if res[k] else 'chưa đạt'}")
    for m, ok, msg in items:
        print(f"   {'✓' if ok else '·'} {m}: {msg}")
if not res["G1"]:
    verdict = "DỪNG LẠI KIỂM TRA: G1 không đạt — quy trình đo có thể có vấn đề"
elif res["G2"]:
    verdict = "GO (gợi ý)" + (" — có cả G3" if res["G3"] else "")
else:
    verdict = "CHƯA ĐỦ: RAG bản mới chữa được phần lớn lỗi — cần bàn lại trọng tâm Trụ 2–3"
print("\nKết luận gợi ý:", verdict)
MANIFEST["go_no_go"] = {k: [dict(model=m, ok=bool(ok), detail=msg) for m, ok, msg in v] for k, v in G.items()}
MANIFEST["verdict_suggested"] = verdict

G1: chưa đạt
   · qwen3-8b: no_rag D1: acc STABLE − acc changed = 0.06 [-0.01, 0.15]
   · llama3.1-8b: no_rag D1: acc STABLE − acc changed = 0.04 [-0.01, 0.12]
G2: ĐẠT
   ✓ qwen3-8b: (a) rag_new recall=1: drift_ext = 0.01 [0.00, 0.02] (n=180) · | (b) mixed − new: Δdrift_ext = 0.78 [0.71, 0.85] ✓
   ✓ llama3.1-8b: (a) rag_new recall=1: drift_ext = 0.06 [0.02, 0.09] (n=180) · | (b) mixed − new: Δdrift_ext = 0.80 [0.73, 0.86] ✓
G3: ĐẠT
   ✓ qwen3-8b: no_rag: Δ(NEW − EXIST) = 0.06 [0.00, 0.13]
   ✓ llama3.1-8b: no_rag: Δ(NEW − EXIST) = 0.27 [0.09, 0.45]

Kết luận gợi ý: DỪNG LẠI KIỂM TRA: G1 không đạt — quy trình đo có thể có vấn đề


## 11. Ghi manifest và đóng gói gửi thầy

`manifest.json` ghi lại:

- cấu hình và mã băm đầu vào, đầu ra;
- phiên bản phần mềm của từng lần chạy (từ các file `meta_*.json`);
- kết luận gợi ý;
- **danh sách mục pilot** (`pilot_item_ids`). Các mục này bị loại khỏi tập test khi báo cáo kết quả cuối cùng.

File `gui_thay.zip` gồm mọi thứ cần để thầy kiểm tra: kết quả đã chấm, các bảng tổng hợp, câu trả lời gốc của mô hình và nhật ký chạy.

In [23]:
MANIFEST["pilot_item_ids"] = sorted({q["item_id"] for q in QUERIES})
MANIFEST["pilot_item_note"] = ("Các mục pilot đã được dùng để ra quyết định thiết kế; "
                               "loại khỏi tập test khi báo cáo kết quả cuối cùng.")
MANIFEST["runs"] = {}
for m in MODELS:
    for cond in CONDITIONS:
        mp = RAW / m["tag"] / f"meta_{cond}.json"
        if mp.exists():
            MANIFEST["runs"][f"{m['tag']}/{cond}"] = json.loads(mp.read_text())
MANIFEST["outputs"] = {str(p.relative_to(OUT)): sha256(p) for p in sorted(OUT.rglob("*"))
                       if p.is_file() and p.suffix in (".csv", ".jsonl") and "raw_smoke" not in p.parts
                       and not p.parts[len(OUT.parts)].startswith("prompts")}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))

ZIP = OUT / "gui_thay.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p != ZIP and not p.parts[len(OUT.parts)].startswith("prompts"):
            z.write(p, p.relative_to(OUT))
print(f"Đã tạo {ZIP.relative_to(ROOT)} ({ZIP.stat().st_size / 1e6:.1f} MB), {len(MANIFEST['pilot_item_ids'])} mục pilot được ghi nhận")

Đã tạo results/pilot-18.1-19.2/gui_thay.zip (0.3 MB), 435 mục pilot được ghi nhận


## 12. Gửi thầy

1. Notebook đã chạy: `04_pilot_llm_rag.ipynb`.
2. File `results/pilot-18.1-19.2/gui_thay.zip`.

Thầy sẽ đọc kết quả, **chấm tay một mẫu câu trả lời** để kiểm tra bộ chấm tự động, rồi cùng em quyết định đi tiếp hay điều chỉnh. Nếu đi tiếp, bước kế là:

- **Trụ 3:** phương pháp nhận biết phiên bản. Ví dụ: kho RAG giữ "phiếu thu hồi" cho mã cũ; truy hồi ưu tiên phiên bản được hỏi; bước hậu kiểm ánh xạ mã cũ sang mã mới.
- **Trụ 1:** đo thẻ lỗi thời ngoài thực tế (luật Sigma, Elastic, CAPEC…).